# Neural CFR+ action sampling on the 30-claim game

This screen asks whether smarter traverser-action sampling becomes useful once full expansion is more expensive.

Each configuration receives equal measured CFR+ training time. Its final learned-average policy is then evaluated by one identically configured action-conditioned fitted-return responder. These responder estimates are discovered lower bounds, not exact exploitability.

The sampled variants all evaluate `CALL` exactly. The structural variants test whether sampling should depend on node width, traversal depth, or predicted cumulative regret.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.training.br_runs import run_best_responder
from liars_poker.training.neural_runs import run_neural_cfr_plus

assert torch.cuda.is_available(), 'This screen requires CUDA.'
device = 'cuda'
torch.set_float32_matmul_precision('high')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
spec = GameSpec(
    ranks=5,
    suits=4,
    hand_size=3,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'Quads'),
    suit_symmetry=True,
)

CFR_MINUTES = 8
TRAVERSALS_PER_PLAYER = 1024
CFR_SEED = 7

BR_MINUTES = 3
BR_SEED = 7
BR_EVALUATE_EVERY_MINUTES = 1
BR_EVAL_EPISODES_PER_ROLE = 200_000
BR_EPISODES_PER_ROLE = 4096
BR_ROLLOUT_BATCH_SIZE = 1024

CONFIGURATIONS = {
    'full expansion': {},
    'cap 16; zero baseline': {
        'traverser_action_sample_count': 16,
        'traverser_action_baseline': 'none',
    },
    'cap 16; CALL baseline': {
        'traverser_action_sample_count': 16,
        'traverser_action_baseline': 'call',
    },
    '75% remaining; CALL baseline': {
        'traverser_action_sample_fraction': 0.75,
        'traverser_action_baseline': 'call',
    },
    'full first, then cap 16; CALL baseline': {
        'traverser_action_sample_count': 16,
        'traverser_action_full_first': True,
        'traverser_action_baseline': 'call',
    },
    'top 8 + random 8; CALL baseline': {
        'traverser_action_sample_count': 16,
        'traverser_action_priority_count': 8,
        'traverser_action_baseline': 'call',
    },
}

BASE_TRAINER_KWARGS = {
    'regret_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': CFR_SEED,
    'regret_buffer_capacity': 500_000,
    'strategy_buffer_capacity': 2_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'regret_train_steps': 24,
    'strategy_train_steps': 6,
    'regret_positive_weight': 0.5,
    'strategy_weighting': 'linear',
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 1024,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}

BR_TRAINER_KWARGS = {
    'state_hidden_sizes': (512, 512),
    'action_hidden_sizes': (128, 128),
    'embedding_dim': 256,
    'device': device,
    'seed': BR_SEED,
    'replay_capacity': 1_000_000,
    'batch_size': 4096,
    'learning_rate': 1e-3,
    'train_steps': 100,
    'warmup_transitions': 20_000,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay_decisions': 500_000,
    'rollouts_per_action': 1,
    'fused_optimizer': True,
}

screen_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
SCREEN_DIR = (
    REPO_ROOT / 'artifacts' / 'neural_method_screens'
    / f'cfr_plus_action_sampling_30_{spec.to_short_str()}___{screen_id}'
)
SCREEN_DIR.mkdir(parents=True, exist_ok=True)
print('claims:', len(rules_for_spec(spec).claims))
print('screen directory:', SCREEN_DIR)
print('planned measured CFR+ minutes:', len(CONFIGURATIONS) * CFR_MINUTES)
print('planned measured responder minutes:', len(CONFIGURATIONS) * BR_MINUTES)

In [ ]:
def safe_name(name):
    return ''.join(ch if ch.isalnum() else '_' for ch in name).strip('_').lower()


def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]


def mean_validation(metrics, section, metric):
    values = [row[metric] for row in metrics[section] if row.get('records', 0)]
    return float(np.mean(values)) if values else np.nan


completed_runs = {}


def load_completed(name, run_dir):
    training_path = run_dir / 'screen_training.pkl'
    br_path = run_dir / 'screen_br_evaluations.csv'
    diagnostics_path = run_dir / 'screen_diagnostics.csv'
    if not all(path.exists() for path in (training_path, br_path, diagnostics_path)):
        return None
    return {
        'training': pd.read_pickle(training_path),
        'br_evaluations': pd.read_csv(br_path),
        'diagnostics': pd.read_csv(diagnostics_path).iloc[0].to_dict(),
    }


def run_configuration(name):
    if name in completed_runs:
        print('Already loaded:', name)
        return completed_runs[name]['br_evaluations']

    run_dir = SCREEN_DIR / safe_name(name)
    cached = load_completed(name, run_dir)
    if cached is not None:
        completed_runs[name] = cached
        print('Loaded completed run:', run_dir)
        return cached['br_evaluations']
    if run_dir.exists() and any(run_dir.iterdir()):
        raise FileExistsError(f'Incomplete run directory: {run_dir}')

    print(f'\n=== CFR+: {name} ===')
    trainer_kwargs = {
        **BASE_TRAINER_KWARGS,
        **CONFIGURATIONS[name],
    }
    cfr_dir = run_dir / 'cfr_plus'
    cfr_result = run_neural_cfr_plus(
        spec,
        minutes=CFR_MINUTES,
        traversals_per_player=TRAVERSALS_PER_PLAYER,
        trainer_kwargs=trainer_kwargs,
        evaluations=[],
        run_dir=cfr_dir,
        save_checkpoint=False,
        wait_for_evaluations=True,
        debug=False,
    )

    training = pd.DataFrame(cfr_result.training_records)
    training['configuration'] = name
    validation = cfr_result.trainer.validation_metrics()
    diagnostics = {
        'configuration': name,
        'CFR+ iterations': int(cfr_result.trainer.iteration),
        'CFR+ measured min': cfr_result.measured_training_s / 60.0,
        'final regret MSE': mean_validation(validation, 'regret', 'mse'),
        'final regret strategy TV': mean_validation(validation, 'regret', 'strategy_tv'),
        'final average-network TV': mean_validation(validation, 'strategy', 'strategy_tv'),
        'run directory': str(run_dir),
    }
    policy_dir = cfr_dir / 'policy'
    del cfr_result
    gc.collect()
    torch.cuda.empty_cache()

    print(f'=== responder: {name} ===')
    br_result = run_best_responder(
        policy_dir,
        method='action_conditioned_fitted_return',
        minutes=BR_MINUTES,
        trainer_kwargs=BR_TRAINER_KWARGS,
        episodes_per_role=BR_EPISODES_PER_ROLE,
        rollout_batch_size=BR_ROLLOUT_BATCH_SIZE,
        evaluate_every_minutes=BR_EVALUATE_EVERY_MINUTES,
        eval_episodes_per_role=BR_EVAL_EPISODES_PER_ROLE,
        exact_evaluation=False,
        run_dir=run_dir / 'approx_br',
        debug=True,
    )
    br_evaluations = pd.DataFrame(br_result.evaluation_records)
    br_evaluations['configuration'] = name
    del br_result
    gc.collect()
    torch.cuda.empty_cache()

    run_dir.mkdir(parents=True, exist_ok=True)
    training.to_pickle(run_dir / 'screen_training.pkl')
    br_evaluations.to_csv(run_dir / 'screen_br_evaluations.csv', index=False)
    pd.DataFrame([diagnostics]).to_csv(run_dir / 'screen_diagnostics.csv', index=False)
    completed_runs[name] = {
        'training': training,
        'br_evaluations': br_evaluations,
        'diagnostics': diagnostics,
    }
    return br_evaluations

In [ ]:
run_configuration('full expansion')

In [ ]:
run_configuration('cap 16; zero baseline')

In [ ]:
run_configuration('cap 16; CALL baseline')

In [ ]:
run_configuration('75% remaining; CALL baseline')

In [ ]:
run_configuration('full first, then cap 16; CALL baseline')

In [ ]:
run_configuration('top 8 + random 8; CALL baseline')

## Results

The responder estimate is the best independently observed first-player response plus the best independently observed second-player response minus one. The conservative lower bound uses the corresponding held-out Wilson bounds.

In [ ]:
summary_rows = []
all_training = []
all_br = []

for name, run in completed_runs.items():
    training = run['training']
    br = run['br_evaluations'].sort_values('measured_training_s')
    timing = pd.json_normalize(training['timing'])
    sampling = pd.json_normalize(training['action_sampling'])
    diagnostics = run['diagnostics']
    all_training.append(training)
    all_br.append(br)

    best_p_first = br['p_first'].max()
    best_p_second = br['p_second'].max()
    best_p_first_lcb = br['p_first_lcb'].max()
    best_p_second_lcb = br['p_second_lcb'].max()
    summary_rows.append({
        'configuration': name,
        'iterations completed': int(training['iteration'].iloc[-1]),
        'iterations / measured min': (
            training['iteration'].iloc[-1]
            / (training['measured_training_s'].iloc[-1] / 60.0)
        ),
        'discovered exploitability estimate': best_p_first + best_p_second - 1.0,
        'conservative lower bound': best_p_first_lcb + best_p_second_lcb - 1.0,
        'mean traversal s': timing['traversal_s'].mean(),
        'mean regret fit s': timing['regret_training_s'].mean(),
        'mean strategy fit s': timing['strategy_training_s'].mean(),
        'mean claim-edge fraction': sampling['claim_edge_fraction'].mean(),
        'mean regret-weight ESS fraction': sampling['regret_weight_ess_fraction'].mean(),
        'largest regret weight': sampling['max_regret_weight'].max(),
        'final regret strategy TV': diagnostics['final regret strategy TV'],
        'final average-network TV': diagnostics['final average-network TV'],
    })

summary = pd.DataFrame(summary_rows).set_index('configuration')
summary = summary.sort_values('discovered exploitability estimate')
display(summary.style.format(precision=6))

training_results = pd.concat(all_training, ignore_index=True)
br_results = pd.concat(all_br, ignore_index=True)
summary.to_csv(SCREEN_DIR / 'summary.csv')
training_results.to_pickle(SCREEN_DIR / 'training_results.pkl')
br_results.to_csv(SCREEN_DIR / 'br_results.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

plot_summary = summary.sort_values('discovered exploitability estimate')
x = np.arange(len(plot_summary))
axes[0].bar(
    x,
    plot_summary['discovered exploitability estimate'],
    label='estimate',
)
axes[0].scatter(
    x,
    plot_summary['conservative lower bound'],
    color='black',
    marker='_',
    s=160,
    label='conservative lower bound',
)
axes[0].set(title='Approximate exploitability', ylabel='Discovered lower-bound estimate')
axes[0].legend(fontsize=8)

axes[1].bar(x, plot_summary['mean traversal s'])
axes[1].set(title='Traversal cost', ylabel='Seconds per iteration')

axes[2].bar(x, plot_summary['mean regret-weight ESS fraction'])
axes[2].set(title='Importance-weight ESS', ylabel='ESS fraction', ylim=(0, 1.05))

for ax in axes:
    ax.set_xticks(x, plot_summary.index, rotation=30, ha='right')
    ax.grid(axis='y', alpha=0.3)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for name, frame in br_results.groupby('configuration', sort=False):
    frame = frame.sort_values('measured_training_s').copy()
    frame['best p_first'] = frame['p_first'].cummax()
    frame['best p_second'] = frame['p_second'].cummax()
    frame['best discovered'] = frame['best p_first'] + frame['best p_second'] - 1.0
    ax.plot(
        frame['measured_training_s'] / 60.0,
        frame['best discovered'],
        marker='o',
        label=name,
    )
ax.set(
    title='Responder compute curves',
    xlabel='Measured responder-training minutes',
    ylabel='Best discovered exploitability estimate',
)
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()

## Reading the screen

A sampled configuration is promising only if it lowers the discovered exploitability estimate at equal CFR+ training time while its responder curve is comparably mature. Higher iterations per minute alone is insufficient.

If `full first, then cap 16` beats the uniform cap, repeated path weighting is the main issue. If `top 8 + random 8` wins, regret-guided allocation is useful despite imperfect regret predictions. If the proportional variant has the best ESS but still loses, local CFR+ clipping noise rather than extreme path weights is likely limiting sampled updates.